In [1]:
import numpy as np
import json

In [2]:
routine_naming_convention = {
    "R1: Scaling -> CNN": "Raw -> Scaled -> CNN",
    "R2: Env -> Scaling -> CNN": "Raw -> Env -> Scaled -> CNN",
    "R3: Scaling -> Sequencing -> LSTM": "Raw -> Scaled -> Sequenced -> LSTM",
    "R4: Env -> Scaling -> Sequencing -> LSTM": "Raw -> Env -> Scaled -> Sequenced -> LSTM",
    "R5: FFT -> Scaling -> CNN": "Raw -> FFT -> Scaled -> CNN",
    "R6: Env -> FFT -> Scaling -> CNN": "Raw -> Env -> FFT -> Scaled -> CNN",
    "R7: FFT -> Scaling -> Resampling -> DNN": "Raw -> FFT -> Scaled -> Resampled -> DNN",
    "R8: Env -> FFT -> Scaling -> Resampling -> DNN": "Raw -> Env -> FFT -> Scaled -> Resampled -> DNN",
    "R9: ZoomedFFT -> Scaling -> CNN": "Raw -> Zoomed FFT -> Scaled -> CNN",
    "R10: Env -> ZoomedFFT -> Scaling -> CNN": "Raw -> Env -> Zoomed FFT -> Scaled -> CNN",
    "R11: ZoomedFFT -> Scaling -> DNN": "Raw -> Zoomed FFT -> Scaled -> DNN",
    "R12: Env -> ZoomedFFT -> Scaling -> DNN": "Raw -> Env -> Zoomed FFT -> Scaled -> DNN",
}
routine_naming_convention_inv = {v: k for k, v in routine_naming_convention.items()}

In [3]:
combinations = {
    "M→C": {
        "source": "mfpt",
        "target": "cwru",
    },
    "M→K": {"source": "mfpt", "target": "kaist"},
    "C→M": {"source": "cwru", "target": "mfpt"},
    "C→K": {"source": "cwru", "target": "kaist"},
    "K→M": {"source": "kaist", "target": "mfpt"},
    "K→C": {"source": "kaist", "target": "cwru"},
}

In [4]:
file_path = "results_fs_supervised_ss_None_sp_None/_fs_supervised_ss_None_sp_None.jsonl"

In [5]:
def jsonl_to_dict(file_path):
    with open(file_path, "r") as f:
        json_list = list(f)

    return {
        json.loads(json_str)["title"]: json.loads(json_str) for json_str in json_list
    }

In [6]:
def mean_training_results_extractor(jsonl_results):

    training_results = {
        "mfpt": {
            "mfpt": [],
            "cwru": [],
            "kaist": [],
        },
        "cwru": {
            "mfpt": [],
            "cwru": [],
            "kaist": [],
        },
        "kaist": {
            "mfpt": [],
            "cwru": [],
            "kaist": [],
        },
    }

    mean_training_results = {
        "mfpt": {},
        "cwru": {},
        "kaist": {},
    }

    for source in jsonl_results.keys():
        for itr in jsonl_results[source]:
            for eval in itr.keys():
                training_results[source][eval].append(itr[eval]["accuracy"])

    for source in training_results.keys():
        for eval in training_results[source].keys():
            mean_training_results[source][eval] = np.mean(
                training_results[source][eval]
            )

    return mean_training_results

In [7]:
def mean_finetuning_results_extractor(jsonl_finetuning_results):

    fine_tuning_results = {
        "mfpt": {
            "cwru": {
                "mfpt": [],
                "cwru": [],
                "kaist": [],
            },
            "kaist": {
                "mfpt": [],
                "cwru": [],
                "kaist": [],
            },
        },
        "cwru": {
            "mfpt": {
                "mfpt": [],
                "cwru": [],
                "kaist": [],
            },
            "kaist": {
                "mfpt": [],
                "cwru": [],
                "kaist": [],
            },
        },
        "kaist": {
            "mfpt": {
                "mfpt": [],
                "cwru": [],
                "kaist": [],
            },
            "cwru": {
                "mfpt": [],
                "cwru": [],
                "kaist": [],
            },
        },
    }

    mean_fine_tuning_results = {
        "mfpt": {
            "cwru": {},
            "kaist": {},
        },
        "cwru": {
            "mfpt": {},
            "kaist": {},
        },
        "kaist": {
            "mfpt": {},
            "cwru": {},
        },
    }

    for src in jsonl_finetuning_results.keys():
        for trg in jsonl_finetuning_results[src].keys():
            for itr in jsonl_finetuning_results[src][trg]:
                for eval in itr.keys():
                    fine_tuning_results[src][trg][eval].append(itr[eval]["accuracy"])
    for src in fine_tuning_results.keys():
        for trg in fine_tuning_results[src].keys():
            for eval in fine_tuning_results[src][trg].keys():
                mean_fine_tuning_results[src][trg][eval] = np.mean(
                    fine_tuning_results[src][trg][eval]
                )

    return mean_fine_tuning_results

In [8]:
def mean_delta_finetuning_results_extractor(
    mean_training_results, mean_finetuning_results
):
    delta_finetuning_results = {
        "mfpt": {
            "cwru": {
                "mfpt": None,
                "cwru": None,
                "kaist": None,
            },
            "kaist": {
                "mfpt": None,
                "cwru": None,
                "kaist": None,
            },
        },
        "cwru": {
            "mfpt": {
                "mfpt": None,
                "cwru": None,
                "kaist": None,
            },
            "kaist": {
                "mfpt": None,
                "cwru": None,
                "kaist": None,
            },
        },
        "kaist": {
            "mfpt": {
                "mfpt": None,
                "cwru": None,
                "kaist": None,
            },
            "cwru": {
                "mfpt": None,
                "cwru": None,
                "kaist": None,
            },
        },
    }

    for src in delta_finetuning_results.keys():
        for trg in delta_finetuning_results[src].keys():
            for eval in delta_finetuning_results[src][trg].keys():
                delta_finetuning_results[src][trg][eval] = (
                    mean_finetuning_results[src][trg][eval]
                    - mean_training_results[src][eval]
                )

    return delta_finetuning_results

In [9]:
def series_extractor(jsonl_dictionary):

    series = {}

    for routine in jsonl_dictionary.keys():
        series[routine] = {
            "base": [],
            "gain": [],
            "drop": [],
        }
        mean_training_results = mean_training_results_extractor(
            jsonl_dictionary[routine]["results"]
        )
        mean_finetuning_results = mean_finetuning_results_extractor(
            jsonl_dictionary[routine]["fine_tuning_results"]
        )
        mean_delta_results = mean_delta_finetuning_results_extractor(
            mean_training_results, mean_finetuning_results
        )
        series[routine]["mean_training"] = mean_training_results
        series[routine]["mean_finetuning"] = mean_finetuning_results
        series[routine]["mean_delta"] = mean_delta_results

        for comb in combinations.keys():
            series[routine]["base"].append(
                mean_training_results[combinations[comb]["source"]][
                    combinations[comb]["target"]
                ]
            )
            series[routine]["gain"].append(
                mean_delta_results[combinations[comb]["source"]][
                    combinations[comb]["target"]
                ][combinations[comb]["target"]]
            )
            series[routine]["drop"].append(
                mean_delta_results[combinations[comb]["source"]][
                    combinations[comb]["target"]
                ][combinations[comb]["source"]]
            )

    return series

In [10]:
diction = jsonl_to_dict(file_path)
series = series_extractor(diction)

print(
    series["Raw -> Zoomed FFT -> Scaled -> DNN"]["base"],
    "\n",
    series["Raw -> Zoomed FFT -> Scaled -> DNN"]["gain"],
    "\n",
    series["Raw -> Zoomed FFT -> Scaled -> DNN"]["drop"],
)

[np.float64(0.4166523605150214), np.float64(0.4414930555555555), np.float64(0.59644128113879), np.float64(0.39184027777777775), np.float64(0.5814946619217082), np.float64(0.32085836909871246)] 
 [np.float64(0.5819742489270385), np.float64(0.5585069444444445), np.float64(0.3772241992882561), np.float64(0.6081597222222223), np.float64(0.3836298932384342), np.float64(0.6784549356223175)] 
 [np.float64(-0.13665480427046273), np.float64(-0.012811387900355853), np.float64(-0.09476394849785408), np.float64(-0.009270386266094333), np.float64(0.0), np.float64(-0.3417824074074074)]


In [11]:
import matplotlib.pyplot as plt


def draw_radar(ax, title, combinations=combinations, base=None, gain=None, drop=None, gain_absolute=False):
    num_vars = len(combinations)
    angles = np.linspace(0, 2 * np.pi, num_vars, endpoint=False).tolist()
    angles_closed = angles + angles[:1]

    if base is not None:
        base_closed = base + base[:1]
        ax.plot(
            angles_closed,
            base_closed,
            color="#fde803",
            linewidth=1.5,
            marker=".",
            label="Base",
        )
        ax.fill(angles_closed, base_closed, color="#fde803", alpha=0.2)

    if gain is not None:
        if gain_absolute:
            gain = [gain[i] + base[i] for i in range(len(gain))]
        gain_closed = gain + gain[:1]
        ax.plot(
            angles_closed,
            gain_closed,
            color="#1f77b4",
            linewidth=1.5,
            marker=".",
            label="Target Gain",
        )
        ax.fill(angles_closed, gain_closed, color="#1f77b4", alpha=0.2)
    if drop is not None:
        drop_closed = drop + drop[:1]

        ax.plot(
            angles_closed,
            np.abs(drop_closed),
            color="#d62728",
            linewidth=1.5,
            marker=".",
            label="Source Drop",
        )
        ax.fill(angles_closed, np.abs(drop_closed), color="#d62728", alpha=0.2)

    ax.set_theta_offset(np.pi / 2)
    ax.set_theta_direction(-1)

    # Tick labels must use the NON-closed angles
    ax.set_thetagrids(np.degrees(angles), combinations, fontsize=14)

    ax.set_ylim(0, 1)
    ax.set_title(title, size=14, pad=15)


def draw_figre(
    series,
    draw_base=True,
    draw_gain=True,
    draw_drop=True,
    show = False,
    export=False,
    export_path=None,
    gain_absolute=False
):

    fig, axes = plt.subplots(
        4,
        3,
        figsize=(14, 18),
        subplot_kw=dict(projection="polar"),
        layout="constrained",
    )

    plt.subplots_adjust(hspace=1, wspace=1)

    for i, ax in enumerate(axes.flatten()):
        r_key = list(routine_naming_convention.keys())[i]

        draw_radar(
            ax,
            r_key,
            base=series[routine_naming_convention[r_key]]["base"]
            if draw_base
            else None,
            gain=series[routine_naming_convention[r_key]]["gain"]
            if draw_gain
            else None,
            drop=series[routine_naming_convention[r_key]]["drop"]
            if draw_drop
            else None,
            gain_absolute=gain_absolute
        )

    from matplotlib.lines import Line2D
    legend_elements = []
    if draw_base:
        legend_elements.append(
            Line2D([0], [0], color="#fde803", lw=2, marker=".", label="Base")
        )

    if draw_gain:
        legend_elements.append(
            Line2D([0], [0], color="#1f77b4", lw=2, marker=".", label="Target Gain")
        )

    if draw_drop:
        legend_elements.append(
            Line2D([0], [0], color="#d62728", lw=2, marker=".", label="Source Drop")
        )

    fig.legend(
        handles=legend_elements,
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=3,
        frameon=False,
        fontsize=12,
    )
    if export:
        plt.savefig(export_path, bbox_inches="tight", dpi=500)
    if show:
        plt.show()


In [12]:
import numpy as np
import pandas as pd

def radar_area(values):
    """
    Compute polygon area of a radar chart.
    """

    values = np.asarray(values)

    n = len(values)

    angles = np.linspace(0, 2 * np.pi, n, endpoint=False)

    x = values * np.cos(angles)
    y = values * np.sin(angles)

    area = 0.5 * np.abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))

    return float(area)


def export_radar_areas(
    series,
    export_path,
    draw_base=True,
    draw_gain=True,
    draw_drop=True,
    gain_absolute=False,
    normalize=False,
):
    """
    Export radar polygon areas for all routines in `series`.

    Parameters
    ----------
    series : dict
        Output of `series_extractor`.

    export_path : str
        CSV export path.

    normalize : bool
        If True, divide by maximum possible polygon area.

    Returns
    -------
    pandas.DataFrame
    """

    rows = []

    for r_key in routine_naming_convention.keys():
        routine_name = routine_naming_convention[r_key]

        data = series[routine_name]

        n = len(data["base"])

        max_area = (n / 2) * np.sin(2 * np.pi / n)

        row = {
            "routine": r_key,
        }

        # BASE
        if draw_base and data.get("base") is not None:
            base_area = radar_area(data["base"])

            if normalize:
                base_area /= max_area

            row["base_area"] = base_area

        # GAIN
        if draw_gain and data.get("gain") is not None:
            gain_values = data["gain"]

            if gain_absolute:
                gain_values = [
                    gain_values[i] + data["base"][i] for i in range(len(gain_values))
                ]

            gain_area = radar_area(gain_values)

            if normalize:
                gain_area /= max_area

            row["gain_area"] = gain_area

        # DROP
        if draw_drop and data.get("drop") is not None:
            drop_values = np.abs(data["drop"])

            drop_area = radar_area(drop_values)

            if normalize:
                drop_area /= max_area

            row["drop_area"] = drop_area

        rows.append(row)

    df = pd.DataFrame(rows)

    df.to_csv(export_path, index=False)

    return df


In [14]:
for file in ["results_fs_dynamic_bootstrapping_gt_recovery_025_ss_None_sp_None/_fs_dynamic_bootstrapping_gt_recovery_025_ss_None_sp_None.jsonl",
             "results_fs_dynamic_bootstrapping_gt_recovery_05_ss_None_sp_None/_fs_dynamic_bootstrapping_gt_recovery_05_ss_None_sp_None.jsonl",
             "results_fs_dynamic_bootstrapping_gt_recovery_075_ss_None_sp_None/_fs_dynamic_bootstrapping_gt_recovery_075_ss_None_sp_None.jsonl",
             "results_fs_dynamic_bootstrapping_gt_recovery_10_ss_None_sp_None/_fs_dynamic_bootstrapping_gt_recovery_10_ss_None_sp_None.jsonl",
             "results_fs_dynamic_bootstrapping_ss_None_sp_None/_fs_dynamic_bootstrapping_ss_None_sp_None.jsonl",
             "results_fs_supervised_ss_None_sp_None/_fs_supervised_ss_None_sp_None.jsonl",
             "results_fs_supervised_ss_percentage_sp_01/_fs_supervised_ss_percentage_sp_01.jsonl",
             "results_fs_supervised_ss_percentage_sp_005/_fs_supervised_ss_percentage_sp_005.jsonl",
             "results_fs_supervised_ss_percentage_sp_001/_fs_supervised_ss_percentage_sp_001.jsonl",
             "results_fs_supervised_ss_shots_per_class_sp_50/_fs_supervised_ss_shots_per_class_sp_50.jsonl",
             "results_fs_supervised_ss_shots_per_class_sp_100/_fs_supervised_ss_shots_per_class_sp_100.jsonl",
             "results_fs_supervised_ss_shots_per_class_sp_150/_fs_supervised_ss_shots_per_class_sp_150.jsonl",
             "results_fs_supervised_ss_shots_per_class_sp_200/_fs_supervised_ss_shots_per_class_sp_200.jsonl",
             "results_fs_supervised_ss_shots_per_class_sp_250/_fs_supervised_ss_shots_per_class_sp_250.jsonl",]:
    
    diction = jsonl_to_dict(file)

    series = series_extractor(diction)

    # draw_figre(
    #     series, export=True, export_path=f"polar_charts/{file.split("/")[-1].split('.')[0]}.png", gain_absolute=True
    # )

    base_name = file.split("/")[-1].split(".")[0]

    export_radar_areas(
        series,
        export_path=f"polar_charts_area/{base_name}_areas.csv",
        gain_absolute=True,
        normalize=True,  # optional
    )